# Experimental-data workflow template

The recommended workflow for applying SFI to **experimental
trajectories**: load a recording, look at it, infer the force with a
generic basis, let PASTIS select the model, and validate the result
without any ground truth — residual diagnostics, a noise-aware refit, a
data-length check and a bootstrap.

The recording is a 2D optical-tweezer track stored as a CSV file.  It is
a *synthetic stand-in* (generated by
``examples/experimental_data/generate_tweezer_csv.py``) with the
imperfections of real tracking data: a finite frame rate and
**localization noise** as large as the bead's displacement between two
frames.  The trap is not quite ideal; the task is to find out how, from
the data alone.  Replace the file path with your own recording to reuse
the template.

**Tags**

synthetic · overdamped · nonlinear · sparsity · diagnostics · 2D

In [ ]:
from __future__ import annotations

from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import random
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter

from SFI.utils.plotting import SFI_COLORS, stamp_output
from SFI.utils import plotting

stamp_output()

## Step 1 — Load the trajectory

`load` reads CSV (with an
optional YAML header), Parquet and HDF5.  This file starts with a
``# dt: 0.01`` header, so the frame interval is set automatically.

In [ ]:
from pathlib import Path
# _here should point to examples/gallery/ in the SFI repository.
# Adjust this path if running from a different location.
_here = Path('examples/gallery') if Path('examples/gallery').exists() else Path('.')
from pathlib import Path

from SFI.trajectory import TrajectoryCollection

# The stand-in recording ships with the SFI examples (path relative to
# examples/gallery/); replace it with your own file to reuse the template.
csv_path = Path("../experimental_data/optical_tweezer.csv")
if not csv_path.exists():           # run from another directory: use the repository layout
    csv_path = _here.parent / "experimental_data" / "optical_tweezer.csv"
coll = TrajectoryCollection.load(csv_path)
dt = coll.dt
print(f"Loaded: {coll.T} frames, d = {coll.d}, dt = {dt}")

## Step 2 — Look at the data

Always look first: outliers, gaps, drifts.  The bead explores a trap
that is narrower along $x$ than along $y$ (left).  At the
frame scale the trace is jittery (middle), and the mean-squared
displacement does not extrapolate to zero at zero lag (right): its
intercept, $2\sigma^2$ per axis, is the classic signature of
localization noise of amplitude $\sigma$.

In [ ]:
_, X_arr, _ = coll.to_arrays(dataset=0)
X_arr = np.asarray(X_arr)[:, 0, :]                  # (T, 2)
t_arr = np.arange(coll.T) * dt
lim = 2.4

def density(X, bins=60):
    H, xe, ye = np.histogram2d(X[:, 0], X[:, 1], bins=bins, range=[[-lim, lim], [-lim, lim]], density=True)
    return gaussian_filter(H.T, 1.0), 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9), width_ratios=[1, 1.4, 1])
H, xc, yc = density(X_arr)
lev = np.linspace(0.04, 1.001, 12) * H.max()
axes[0].contourf(xc, yc, H, levels=lev, cmap="Blues")
axes[0].set_aspect("equal")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].set_title("Where the bead goes")

n_zoom = 300
axes[1].plot(t_arr[:n_zoom], X_arr[:n_zoom, 0], lw=0.8, color=SFI_COLORS["data"], label="$x$")
axes[1].plot(t_arr[:n_zoom], X_arr[:n_zoom, 1], lw=0.8, color=SFI_COLORS["exact"], label="$y$")
axes[1].set_xlabel("time"); axes[1].set_ylabel("position")
axes[1].set_title(f"First {n_zoom} frames")
axes[1].legend(loc="upper right", fontsize=8)

lags = np.arange(1, 11)
msd = np.array([np.mean((X_arr[k:] - X_arr[:-k]) ** 2, axis=0) for k in lags])   # (lags, 2)
for a, (c, lab) in enumerate([(SFI_COLORS["data"], "$x$"), (SFI_COLORS["exact"], "$y$")]):
    slope, icpt = np.polyfit(lags[:4] * dt, msd[:4, a], 1)
    tt = np.linspace(0, lags[-1] * dt, 50)
    axes[2].plot(lags * dt, msd[:, a], "o", ms=5, color=c, label=f"{lab}: $\\sigma \\approx$ {np.sqrt(icpt / 2):.2f}")
    axes[2].plot(tt, icpt + slope * tt, "--", lw=1, color=c)
axes[2].set_xlim(0, None); axes[2].set_ylim(0, None)
axes[2].set_xlabel(r"lag $\tau$"); axes[2].set_ylabel(r"MSD$(\tau)$")
axes[2].set_title("Short-lag MSD: noise floor")
axes[2].legend(loc="lower right", fontsize=8)
plt.show()

## Step 3 — Fast linear inference and model selection

Start with a generic polynomial basis — all monomials up to degree 3
for each force component, 20 candidate terms — and the fast linear
estimators.  With default settings (``"auto"``) SFI detects the
measurement noise and switches both the diffusion estimate and the
force fit to their noise-robust variants.  PASTIS then selects the
smallest model the data support.

In [ ]:
from SFI import OverdampedLangevinInference
from SFI.bases import monomials_up_to

B = monomials_up_to(order=3, dim=2, rank="vector")
labels = list(B.labels)

inf_lin = OverdampedLangevinInference(coll)
inf_lin.compute_diffusion_constant()
inf_lin.infer_force_linear(B)
inf_lin.compute_force_error()
inf_lin.sparsify_force(criterion="PASTIS")
inf_lin.compute_force_error()
inf_lin.print_report()

## Step 4 — Residual diagnostics

Without ground truth, the first check is statistical: are the
standardized residuals of the fit consistent with white Gaussian
noise?  Each flag, if any, comes with a one-line hint.

In [ ]:
report_lin = inf_lin.diagnose()
report_lin.print_summary()

The linear fit passes.  But clean residual tests are necessary, not
sufficient: they see missing *structure* (autocorrelation,
non-Gaussianity), while a smooth bias of the coefficients — the typical
effect of localization noise on a nonlinear basis — leaves the
residuals white.  The decisive cross-check is an independent estimator
built for this situation.

## Step 5 — Noise-aware parametric refit

The parametric estimator
`infer_force` models
the localization noise explicitly: it fits the force, the diffusion and
the noise covariance $\Lambda$ jointly, so no separate diffusion
call is needed.  It is slower than the linear route, and it is the
estimator to report for noisy recordings
(:doc:`/inference/noise_and_sampling`).

In [ ]:
inf = OverdampedLangevinInference(coll)
inf.infer_force(B)
inf.compute_force_error()
inf.sparsify_force(criterion="PASTIS")
inf.compute_force_error()
inf.print_report()

sigma_hat = np.sqrt(np.diag(np.asarray(inf.Lambda)))
print(f"Estimated localization noise: sigma = {sigma_hat.round(3)} per axis")

inf.diagnose().print_summary()

The noise amplitude matches the MSD noise floor of step 2, and the
residuals pass again.  Left: the Pareto front of the parametric fit —
the information gained by the best model of each size; it flattens
once five terms are in, where PASTIS stops (AIC, less strict, would
admit two more).  Right: the selected coefficients of both routes with
95% error bars.  Where the two estimators disagree by more than their
errors, the linear estimate is the one biased by the noise.

In [ ]:
support = [int(i) for i in np.asarray(inf.force_support)]
union = sorted(set(support) | {int(i) for i in np.asarray(inf_lin.force_support)})

# full-basis coefficients and covariance of each sparse fit (pruned terms are 0)
(c_par, cov_par), (c_lin, cov_lin) = inf.coeff_block(B), inf_lin.coeff_block(B)
c_par, e_par = c_par[union], np.sqrt(np.diag(cov_par))[union]
c_lin, e_lin = c_lin[union], np.sqrt(np.diag(cov_lin))[union]

term_tex = {lab: lab for lab in labels}
term_tex.update({"x0·e0": r"$x\,\hat e_x$", "x1·e1": r"$y\,\hat e_y$", "x1·e0": r"$y\,\hat e_x$",
                 "x0·e1": r"$x\,\hat e_y$", "x0^3·e0": r"$x^3\,\hat e_x$"})
tick_labels = [term_tex[labels[i]] for i in union]

def coef_panel(ax, truth=None):
    pos = np.arange(len(union))
    for off, c, e, col, lab in [(-0.12, c_lin, e_lin, SFI_COLORS["highlight"], "linear"),
                                (0.12, c_par, e_par, SFI_COLORS["inferred"], "parametric")]:
        ax.errorbar(pos + off, c, yerr=1.96 * e, fmt="o", ms=6, capsize=4, color=col, ecolor=col, label=lab)
    if truth is not None:
        ax.scatter(pos, truth, marker="_", s=500, lw=2.5, color=SFI_COLORS["exact"], zorder=4, label="generator")
    ax.axhline(0, color="#808080", lw=0.5)
    ax.set_xticks(pos, tick_labels, fontsize=11)
    ax.set_ylabel("coefficient (± 95%)")
    ax.legend(fontsize=8, loc="upper right")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
plotting.plot_pareto_front(inf.force_sparsity_result, criteria=("PASTIS", "AIC"), ax=axes[0])
n_sizes = int(np.sum(np.asarray(inf.force_sparsity_result.best_info_by_k) > -1e30))
axes[0].set_xlim(-0.5, n_sizes - 0.5)
axes[0].set_title("Pareto front (parametric fit)")
coef_panel(axes[1])
axes[1].set_title("Selected terms, both estimators")
plt.show()

Read as physics: $x\,\hat e_x$ and $y\,\hat e_y$ are the
trap stiffnesses; the antisymmetric pair $y\,\hat e_x$,
$x\,\hat e_y$ is a rotation — a non-conservative force, the
signature of a non-ideal tweezer; and $x^3\,\hat e_x$ is an
anharmonicity along $x$.

## Step 6 — Is there enough data?

Refit the selected model on the first quarter and the first half of the
recording.  A trustworthy model has coefficients that stay put within
their error bars while the error bars shrink as $1/\sqrt{T}$;
terms that drift or flip sign as data are added are not yet resolved.

In [ ]:
conv = {}
for frac in (0.25, 0.5):
    first_part, _ = coll.split_time(frac)        # the first `frac` of the recording
    inf_f = OverdampedLangevinInference(first_part)
    inf_f.infer_force(B[support])               # the five selected terms only
    inf_f.compute_force_error()
    conv[frac] = (np.asarray(inf_f.force_coefficients), np.asarray(inf_f.force_coefficients_stderr))
conv[1.0] = (np.asarray(inf.force_coefficients), np.asarray(inf.force_coefficients_stderr))
for frac, (c, e) in conv.items():
    print(f"{frac:4.0%} of the data: " + ", ".join(
        f"{labels[i]} = {ci:+.2f} ± {ei:.2f}" for i, ci, ei in zip(support, c, e)))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
cols = [SFI_COLORS[k] for k in ("data", "highlight", "exact", "inferred", "tertiary")]
fr = np.array(sorted(conv))
for j, i in enumerate(support):
    vals = np.array([conv[f][0][j] for f in fr])
    errs = np.array([conv[f][1][j] for f in fr])
    ax.errorbar(fr * coll.T * dt * (1 + 0.04 * (j - 2)), vals, yerr=1.96 * errs, fmt="o-", ms=5,
                capsize=3, color=cols[j], label=term_tex[labels[i]])
ax.axhline(0, color="#808080", lw=0.5)
ax.set_xscale("log")
ax.set_xticks(fr * coll.T * dt, [f"{f * coll.T * dt:.0f}" for f in fr])
ax.minorticks_off()
ax.set_xlabel("observation time used")
ax.set_ylabel("coefficient (± 95%)")
ax.set_title("Convergence with data length")
ax.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.22))
plt.show()

All five terms are resolved from the first quarter of the recording on,
and no estimate moves by more than its 95% error bar.

## Step 7 — Bootstrap

Finally, simulate new recordings from the inferred model — force,
diffusion, and the estimated localization noise added on top — and
compare them with the data, on statistics the fit never targeted: the
stationary distribution, and the antisymmetric cross-correlation


$$
A(\tau) = \langle x(t)\,y(t+\tau)\rangle - \langle y(t)\,x(t+\tau)\rangle ,
$$

which vanishes for any equilibrium (conservative) trap: it measures how
much the rotation breaks time-reversal symmetry.  Twenty bootstraps
give the spread expected for a recording of this length.

In [ ]:
coll_boot, proc_boot = inf.simulate_bootstrapped_trajectory(random.PRNGKey(123))
boots = [coll_boot] + [proc_boot.simulate(dt=dt, Nsteps=coll.T, key=random.PRNGKey(s)) for s in range(19)]
boots = [b.degrade(noise=float(sigma_hat.mean()), seed=s) for s, b in enumerate(boots)]

n_lag = int(3.0 / dt)
tau = np.arange(n_lag) * dt

def antisym(X):
    X = X - X.mean(axis=0)
    x, y = X[:, 0], X[:, 1]
    out = np.empty(n_lag)
    out[0] = 0.0
    for k in range(1, n_lag):
        out[k] = np.mean(x[:-k] * y[k:]) - np.mean(y[:-k] * x[k:])
    return out

Xb_all = [np.asarray(b.to_arrays(dataset=0)[1])[:, 0, :] for b in boots]
A_data = antisym(X_arr)
A_boot = np.array([antisym(Xb) for Xb in Xb_all])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), width_ratios=[1, 1.5])
Hb, _, _ = density(np.concatenate(Xb_all))
lev_b = np.linspace(0.08, 0.95, 6) * H.max()
axes[0].contourf(xc, yc, H, levels=np.r_[lev_b, 1.01 * H.max()], cmap="Blues", alpha=0.9)
axes[0].contour(xc, yc, Hb, levels=lev_b, colors=SFI_COLORS["bootstrap"], linewidths=1.6)
axes[0].set_aspect("equal")
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$y$")
axes[0].set_title("Stationary density")
axes[0].legend(handles=[Patch(color="#6baed6", label="data"),
                        Line2D([], [], color=SFI_COLORS["bootstrap"], lw=1.6, label="bootstrap")],
               fontsize=8, loc="upper right")

lo, hi = A_boot.min(axis=0), A_boot.max(axis=0)
axes[1].fill_between(tau, lo, hi, color=SFI_COLORS["bootstrap"], alpha=0.5, lw=0,
                     label=f"bootstrap range ({len(boots)} runs)")
axes[1].plot(tau, A_data, color=SFI_COLORS["data"], lw=2, label="data")
axes[1].axhline(0, color="#808080", lw=0.8, ls=":", label="any equilibrium trap")
axes[1].set_xlabel(r"lag $\tau$")
axes[1].set_ylabel(r"$A(\tau)$")
axes[1].set_title("Broken time-reversal symmetry")
axes[1].legend(fontsize=8, loc="upper right")
plt.show()

The recording sits inside the bootstrap spread on both counts: the
inferred model reproduces the shape of the trap and the strength and
lifetime of its circulation.

## Checking against the generator

This recording is synthetic, so — unlike with real data — we can grade
the workflow.  The generator uses a harmonic trap of stiffness
$k = 1$, a cubic term $-x^3$ along $x$, a rotation of
rate $\omega = 0.5$, $D = 0.5$ and a localization noise of
$0.1$ per axis.

In [ ]:
theta_true = {"x0·e0": -1.0, "x1·e1": -1.0, "x1·e0": -0.5, "x0·e1": 0.5, "x0^3·e0": -1.0}
truth = np.array([theta_true.get(labels[i], 0.0) for i in union])
print("term       true    parametric (z)          linear (z)")
for j, i in enumerate(union):
    print(f"{labels[i]:9s} {truth[j]:+.2f}   {c_par[j]:+.3f} ± {e_par[j]:.3f} ({(c_par[j] - truth[j]) / e_par[j]:+.1f})"
          f"   {c_lin[j]:+.3f} ± {e_lin[j]:.3f} ({(c_lin[j] - truth[j]) / e_lin[j]:+.1f})")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
coef_panel(ax, truth=truth)
ax.set_title("Both estimators vs the generator")
plt.show()

Both routes find the right five terms.  The parametric coefficients
deviate from the generator by what sampling noise allows: the largest
deviation, about two standard errors on the $y$ stiffness, is
shared by the linear estimate, so it is a fluctuation of this recording
rather than a bias.  The linear ones are biased beyond their error bars on the terms the noise
enters nonlinearly — the $x$ stiffness and the cubic, which trade
off against each other — even though their residual diagnostics looked
clean.  On real data, that is exactly what the step-5 comparison is
for.

## Thumbnail

In [ ]:
fig_thumb = plt.figure(figsize=(4, 4))
ax_t = fig_thumb.add_subplot(111)
ax_t.contourf(xc, yc, H, levels=lev, cmap="Blues")
gx, gy = np.meshgrid(np.linspace(-2, 2, 40), np.linspace(-2, 2, 40))
Fg = np.asarray(inf.force_inferred(jnp.asarray(np.stack([gx.ravel(), gy.ravel()], axis=1))))
ax_t.streamplot(gx, gy, Fg[:, 0].reshape(gx.shape), Fg[:, 1].reshape(gx.shape), density=1.1,
                color=SFI_COLORS["inferred"], linewidth=1.0, arrowsize=0.8)
ax_t.set_aspect("equal")
ax_t.set_xlim(-2, 2); ax_t.set_ylim(-2, 2)
ax_t.set_xticks([]); ax_t.set_yticks([])
ax_t.set_xlabel(""); ax_t.set_ylabel("")
plt.show()